In [1]:
import jax.random as random
import jax.numpy as jnp
from models import IQPTensorNetwork, local_gates
nqubits = 5
gate_list = local_gates(nqubits, max_weight=3)
iqp = IQPTensorNetwork(nqubits=nqubits, interactions=gate_list)

key = random.PRNGKey(42)
params = random.uniform(key, shape=(len(gate_list),), minval=0, maxval=2*jnp.pi)

circuit = iqp.build_circuit(parameters=params)

In [3]:
from expectation import expvals_sampling, expvals_mc

expvals_sampling(circuit, gate_list, 10)

(Array([-0.4, -0.2, -0.2,  0. ,  0.2,  0.4,  0.4, -0.2,  0.4,  0.2,  0. ,
         0.2,  0.4, -0.2,  0. , -0.4, -0.6,  0. , -0.2,  0.4, -0.2, -0.4,
        -0.2, -0.4, -0.4], dtype=float32),
 Array([0.30550504, 0.32659867, 0.3265986 , 0.3333333 , 0.3265986 ,
        0.30550504, 0.30550504, 0.32659867, 0.30550504, 0.32659867,
        0.3333333 , 0.3265986 , 0.30550504, 0.32659867, 0.3333333 ,
        0.30550504, 0.26666668, 0.3333333 , 0.32659867, 0.30550504,
        0.3265986 , 0.30550504, 0.32659867, 0.30550504, 0.30550504],      dtype=float32))

In [6]:
import jax.numpy as jnp
import jax
import numpy as np
from iqpopt import IqpSimulator
from iqpopt.utils import local_gates

def convert_to_binary_matrix(gate_list: list, n_qubits: int) -> jnp.ndarray:
    """
    Manually convert a list of gate indices into a binary generator matrix 
    compatible with IQPOptimizer.
    """
    n_generators = len(gate_list)
    matrix = np.zeros((n_generators, n_qubits), dtype=int)
    
    for i, gate in enumerate(gate_list):
        for qubit_idx in gate:
            matrix[i, qubit_idx] = 1
            
    return jnp.array(matrix)


n_qubits = 4
max_weight = 2

quimb_gates = local_gates(n_qubits, max_weight) # this is iqpopt function!! with 1 more nesting layer
print(f"Number of generators: {len(quimb_gates)}")
generators_matrix = convert_to_binary_matrix(quimb_gates, n_qubits)

iqp = IqpSimulator(
    n_qubits=n_qubits,
    gates=quimb_gates,
)

def test_consistency():
    key = jax.random.PRNGKey(0)
    params = jax.random.uniform(key, (len(quimb_gates),))
    ops = generators_matrix[0:10]
    
    res_iqp, _ = iqp.op_expval_batch(params, ops, n_samples=1000, key=key)    
    res_manual, _ = expvals_mc(params, ops, generators_matrix, n_samples=1000, key=key)
    
    np.testing.assert_allclose(res_iqp, res_manual, atol=1e-7)
    print("✅ Conversione e campionamento coerenti!")

if __name__ == "__main__":
    test_consistency()

Number of generators: 10
✅ Conversione e campionamento coerenti!
